# 실시간 추론 및 모델 모니터링을 통한 ML 실험

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ 호환성 안내:</strong> 이 노트북은 <strong>SageMaker Distribution Image 4.0.0</strong> 및 <strong>SageMaker Python SDK 버전 3.7.1</strong>에서 테스트되었습니다.
</div>

## 개요
이 노트북은 [Amazon SageMaker AI](https://docs.aws.amazon.com/sagemaker/latest/dg/whatis.html)를 사용한 엔드투엔드 머신러닝 워크플로우를 시연합니다:

- **모델 훈련**: SageMaker [ModelTrainer (SDK v3)](https://sagemaker.readthedocs.io/en/stable/training/index.html)를 사용하여 XGBoost 모델 훈련
- **MLflow 통합**: [SageMaker AI MLflow 앱](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)에 실험 추적 및 아티팩트 로깅
- **실시간 엔드포인트**: [데이터 캡처](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-data-capture.html)가 활성화된 SageMaker [실시간 엔드포인트](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints.html) 배포
- **데이터 드리프트 모니터링**: [Evidently (오픈소스)](https://www.evidentlyai.com/evidently-oss)를 사용하여 캡처된 데이터의 데이터 드리프트 감지
- **모델 품질 모니터링**: Evidently (오픈소스)를 사용하여 모델 품질 보고서 생성
- **MLflow의 드리프트 메트릭**: 추적을 위해 드리프트 메트릭과 보고서를 MLflow에 로깅

||||
|---|---|---|
|1. |노트북에서 실험하기 ||
|2. |SageMaker AI 처리 작업과 SageMaker SDK로 확장하기 ||
|3. |ML 파이프라인, 모델 레지스트리, 피처 스토어로 운영화하기 ||
|4. |모델 빌드 CI/CD 파이프라인 추가하기 ||
|5. |모델 배포 파이프라인 추가하기 ||
|6. |모델 및 데이터 모니터링 추가하기 |**<<<< 현재 위치**|

## 워크플로우
아래 다이어그램은 실시간 추론 엔드포인트를 사용한 전체 모니터링 워크플로우를 보여줍니다.

1. [훈련 작업](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-training.html)에서 모델을 훈련하고, 훈련된 모델의 기준 데이터셋을 S3에 저장합니다.
2. 데이터 캡처가 활성화된 실시간 엔드포인트에 모델을 배포합니다; 데이터 캡처는 엔드포인트의 입력과 출력을 S3에 저장합니다.
3. Evidently를 사용하여 기준 데이터셋과 엔드포인트에서 캡처된 데이터를 기반으로 데이터 드리프트와 모델 품질을 계산합니다.
4. 추적 및 비교를 위해 데이터 드리프트와 모델 품질 보고서를 MLflow에 저장합니다.
5. 선택적으로 데이터 또는 모델 드리프트가 특정 임계값을 초과하면 알림을 트리거합니다.

![실시간 엔드포인트를 사용한 모델 모니터링 워크플로우](img/arch-sagemaker-inference-predictiveml-monitoring-RealTime-inf.png "실시간 엔드포인트를 사용한 모델 모니터링 워크플로우")

### 비즈니스 문제
이 샘플은 UCI 머신러닝 저장소의 [은행 마케팅](https://archive.ics.uci.edu/dataset/222/bank+marketing) 데이터셋을 사용합니다. 다음을 기반으로 은행 고객이 정기 예금에 가입할지 예측하는 데 사용할 수 있습니다:
- 인구통계 (나이, 직업, 혼인 상태, 교육)
- 재무 정보 (신용 불이행, 주택 대출, 개인 대출)
- 캠페인 데이터 (연락 유형, 월, 요일, 통화 시간)
- 경제 지표 (고용률, 소비자 물가 지수 등)

---

<div style="border: 4px solid #d32f2f; border-radius: 10px; padding: 20px; background-color: #ffebee; margin: 15px 0;">

### ⛔ 중요: Python 3.10 필수

이 워크샵의 모든 노트북은 **Python 3.10**에서 실행해야 합니다. 다른 Python 버전을 사용하면 아래 셀이 `AssertionError`로 실패합니다.

오류가 발생하면 다음 단계를 따르세요:

1. JupyterLab에서 **터미널**을 엽니다 (**File → New → Terminal**)
2. 설정 스크립트를 실행합니다:
   ```bash
   cd ~/amazon-sagemaker-from-idea-to-production
   ./setup-py310.sh
   ```
3. 새 커널을 적용하려면 **브라우저를 새로고침**합니다
4. 이 노트북에서 **Kernel → Change Kernel → Python 3.10**으로 이동합니다
5. **커널을 재시작**하고 처음부터 다시 실행합니다

</div>

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;"> 
<h3>⚠️ 인위적 드리프트가 포함된 시뮬레이션된 프로덕션 데이터</h3>
<p><strong>목적:</strong> 이 노트북은 프로덕션 배포가 아닌 드리프트 감지 기능을 시연하기 위해 사용합니다</p>

In [ ]:
import sys

assert sys.version_info[:2] == (3, 10), f"Expected Python 3.10, got {sys.version_info.major}.{sys.version_info.minor}. Please make sure you run on Python 3.10"

## 1: 사전 요구사항 및 설정

필요한 패키지를 설치하고 라이브러리를 임포트합니다.

In [ ]:
%store -r 

%store

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

In [ ]:
import boto3
import sagemaker
import mlflow
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import zipfile
import os
import json
import time
from datetime import datetime

from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.core.training.configs import SourceCode, InputData, Compute
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.serve.builder.schema_builder import SchemaBuilder
from sagemaker.core.transformer import Transformer

from evidently import Report, Dataset, DataDefinition, BinaryClassification
from evidently.presets import DataDriftPreset, DataSummaryPreset, ClassificationPreset
from evidently.metrics import *

print(f'MLflow version: {mlflow.__version__}')

In [ ]:
sagemaker_session = Session()

print('AWS Configuration:')
print(f'  Region: {region}')
print(f'  S3 Bucket: {bucket_name}')
print(f'  IAM Role: {sm_role}')
print(f'  Data Prefix: {bucket_prefix}')

---

## 2: SageMaker AI MLflow 앱 설정

### SageMaker AI MLflow 앱이란?

다음을 제공하는 오픈소스 MLflow 기반의 완전 관리형 서비스입니다:
- **실험 추적**: 파라미터, 메트릭, 아티팩트 로깅
- **모델 레지스트리**: 모델 버전 관리
- **SageMaker 통합**: SageMaker AI 모델 레지스트리에 자동 등록
- **재현성**: 코드 버전 및 의존성 추적
- **협업**: 팀 간 실험 공유

### 지침

1. SageMaker Studio에서 왼쪽 사이드바로 이동합니다
2. **애플리케이션** 아래의 "MLflow" 패널을 클릭합니다
3. MLflow 앱(보통 "DefaultMLFlowApp"으로 명명)을 찾거나 필요한 경우 새로 생성합니다
4. 앱 이름을 복사하고 특정 앱을 사용해야 하는 경우 아래 셀에서 편집합니다

아래 셀은 MLflow 앱이 존재하는지 확인하고 해당 ARN을 가져옵니다.

이전 셀에서 가져온 ARN이 기본 MLflow 추적 URI로 설정됩니다. 필요한 경우 MLflow에서 새 실험이 생성되고, 그렇지 않으면 기존 실험이 기본값으로 설정됩니다.

In [ ]:
mlflow.set_tracking_uri(mlflow_arn)
mlflow_experiment_name = 'demo-ml-xgb-train-monitor'
mlflow_model_name = 'xgb-ml'

try:
    # Try to create a new experiment
    experiment_id = mlflow.create_experiment(mlflow_experiment_name)
    print(f'Created new MLflow experiment: {mlflow_experiment_name}')
    print(f'Experiment ID: {experiment_id}')
except:
    # Experiment already exists, set it as active
    mlflow.set_experiment(mlflow_experiment_name)
    experiment = mlflow.get_experiment_by_name(mlflow_experiment_name)
    print(f'Using existing MLflow experiment: {mlflow_experiment_name}')
    print(f'Experiment ID: {experiment.experiment_id}')

---

## 3: 데이터 준비
이 섹션에서는 은행 마케팅 데이터셋을 다운로드하고 XGBoost 모델 훈련을 위한 데이터셋을 준비하는 간단한 전처리 단계를 적용합니다.

데이터를 pandas DataFrame에 로드하고 타겟 분포를 빠르게 확인합니다.

In [ ]:
df = pd.read_csv('data/bank-additional/bank-additional-full.csv', sep=';')
print(f'Dataset shape: {df.shape}')
print(f'\nTarget distribution:')
print(df['y'].value_counts())
print(f'\nFirst few rows:')
df.head()

모든 범주형 피처와 타겟 변수를 인코딩합니다.

In [ ]:
cat_cols = [c for c in df.select_dtypes(include=["object", "string"]).columns if c != 'y']
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

df['y'] = (df['y'] == 'yes').astype(int)
print(f'Encoded {len(cat_cols)} categorical features')
print(f'Feature names: {df.columns.tolist()}')

이제 층화 샘플링으로 데이터를 훈련(70%), 검증(15%), 테스트(15%) 세트로 분할합니다.

훈련 데이터셋은 두 번 저장됩니다: 한 번은 모델 훈련용으로, 한 번은 데이터 드리프트 계산을 위한 기준 데이터셋으로 저장됩니다.

In [ ]:
X, y = df.drop('y', axis=1), df['y']

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Second split: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f'Training set: {X_train.shape}')
print(f'Validation set: {X_val.shape}')
print(f'Test set: {X_test.shape}')

os.makedirs('data', exist_ok=True)
pd.concat([y_train, X_train], axis=1).to_csv('data/train.csv', index=False, header=False)
pd.concat([y_val, X_val], axis=1).to_csv('data/validation.csv', index=False, header=False)
pd.concat([y_test, X_test], axis=1).to_csv('data/test.csv', index=False, header=False)

# Save baseline (reference) data for drift monitoring - use training data features
X_train.to_csv('data/baseline.csv', index=False, header=True)
print('Data split and saved locally')

모든 데이터셋을 S3에 업로드합니다.

In [ ]:
train_s3 = sagemaker_session.upload_data('data/train.csv', bucket_name, f'{bucket_prefix}/data/train')
validation_s3 = sagemaker_session.upload_data('data/validation.csv', bucket_name, f'{bucket_prefix}/data/validation')
test_s3 = sagemaker_session.upload_data('data/test.csv', bucket_name, f'{bucket_prefix}/data/test')
baseline_s3 = sagemaker_session.upload_data('data/baseline.csv', bucket_name, f'{bucket_prefix}/data/baseline')

print(f'Train S3: {train_s3}')
print(f'Validation S3: {validation_s3}')
print(f'Test S3: {test_s3}')
print(f'Baseline S3: {baseline_s3}')

---
## 4: MLflow 추적을 통한 모델 훈련

MLflow 통합과 함께 SageMaker ModelTrainer를 사용하여 [XGBoost 모델](https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html)을 훈련합니다.

### 주요 기능:
- **ModelTrainer**: 훈련을 위한 간소화된 SageMaker SDK v3 API
- **MLflow 자동 로깅**: 파라미터, 메트릭, 모델 아티팩트를 자동으로 캡처
- **모델 레지스트리**: MLflow 및 SageMaker 모델 레지스트리에 자동 등록

훈련 스크립트와 요구사항 파일을 저장할 디렉토리를 생성합니다.

In [ ]:
os.makedirs('scripts', exist_ok=True)

In [ ]:
%%writefile scripts/train.py
import argparse
import os
import json
import logging
import sys
import xgboost as xgb
import pandas as pd
import mlflow
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
logger.addHandler(logging.StreamHandler(sys.stdout))


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--max_depth', type=int, default=5)
    parser.add_argument('--eta', type=float, default=0.2)
    parser.add_argument('--gamma', type=int, default=4)
    parser.add_argument('--min_child_weight', type=int, default=6)
    parser.add_argument('--subsample', type=float, default=0.8)
    parser.add_argument('--num_round', type=int, default=100)
    return parser.parse_known_args()

if __name__ == '__main__':
    args, _ = parse_args()
    
    train_data = pd.read_csv('/opt/ml/input/data/train/train.csv', header=None)
    val_data = pd.read_csv('/opt/ml/input/data/validation/validation.csv', header=None)
    
    X_train, y_train = train_data.iloc[:, 1:], train_data.iloc[:, 0]
    X_val, y_val = val_data.iloc[:, 1:], val_data.iloc[:, 0]
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    
    mlflow_app_arn = os.environ.get('MLFLOW_TRACKING_URI', None)
    mlflow_experiment_name = os.environ.get('MLFLOW_EXP', None)
    mlflow_model_name = os.environ.get('MLFLOW_MODEL_NAME', None)
    mlflow.set_tracking_uri(mlflow_app_arn)
    mlflow.set_experiment(mlflow_experiment_name)
    
    mlflow.xgboost.autolog(
        log_input_examples=True,
        log_model_signatures=True,
        log_models=True,
        log_datasets=True,
        model_format="json",
        registered_model_name=mlflow_model_name,
        extra_tags={"team": "data-science", "use_case": "bank-marketing"},
    )
    
    with mlflow.start_run(run_name=f"training_{mlflow_model_name}"):
        params = {
            'max_depth': args.max_depth,
            'eta': args.eta,
            'gamma': args.gamma,
            'min_child_weight': args.min_child_weight,
            'subsample': args.subsample,
            'objective': 'binary:logistic',
            'eval_metric': 'auc'
        }
        
        mlflow.log_params(params)
        model = xgb.train(params, dtrain, args.num_round, evals=[(dval, 'validation')])
        
        y_pred_proba = model.predict(dval)
        y_pred = (y_pred_proba > 0.5).astype(int)
        
        metrics = {
            'Accuracy': accuracy_score(y_val, y_pred),
            'Precision': precision_score(y_val, y_pred),
            'Recall': recall_score(y_val, y_pred),
            'F1Score': f1_score(y_val, y_pred),
            'auc': roc_auc_score(y_val, y_pred_proba)
        }
        
        mlflow.log_metrics(metrics)
        print(f'Validation Metrics: {metrics}')
        
        model_path = '/opt/ml/model'
        os.makedirs(model_path, exist_ok=True)
        model.save_model(f'{model_path}/{mlflow_model_name}')
        mlflow.xgboost.log_model(
            model, 
            artifact_path="model",
            registered_model_name=mlflow_model_name
        )

SageMaker는 [XGBoost 컨테이너](https://github.com/aws/sagemaker-xgboost-container)를 포함한 다양한 프레임워크를 위한 사전 빌드된 컨테이너를 제공합니다. 이전에 생성된 요구사항 파일은 아래에서 가져오는 사전 빌드된 컨테이너에 추가 패키지를 설치하는 데 사용됩니다.

In [ ]:
xgboost_image
print(f'XGBoost training image: {xgboost_image}')

ModelTrainer를 사용하여 이전에 생성된 훈련 스크립트, 사전 빌드된 컨테이너, 특정 컴퓨팅 설정, 하이퍼파라미터 세트로 SageMaker 훈련 작업을 설정합니다. MLflow 설정은 환경 변수를 통해 전달되어 훈련 작업이 MLflow 앱에 메트릭과 파라미터를 로깅할 수 있습니다.

In [ ]:
hyperparameters = {
    'max_depth': 5,
    'eta': 0.2,
    'gamma': 4,
    'min_child_weight': 6,
    'subsample': 0.8,
    'num_round': 100
}

model_trainer = ModelTrainer(
    training_image=xgboost_image,
    source_code=SourceCode(
        source_dir='scripts',
        entry_script='train.py',
    ),
    compute=Compute(
        instance_type='ml.r5.xlarge',
        instance_count=1,
        volume_size_in_gb=30
    ),
    hyperparameters=hyperparameters,
    base_job_name='bank-marketing-xgboost',
    environment={
        'MLFLOW_TRACKING_URI': mlflow_arn,
        'MLFLOW_EXP': mlflow_experiment_name,
        'MLFLOW_MODEL_NAME': mlflow_model_name
    }
)

훈련 및 검증 데이터셋으로 훈련 작업을 시작합니다. 완료하는 데 약 5~7분이 소요됩니다.

In [ ]:
%%time
input_data_train = InputData(channel_name='train', data_source=train_s3)
input_data_validation = InputData(channel_name='validation', data_source=validation_s3)

print('Starting model training...')

model_trainer.train(
    input_data_config=[input_data_train, input_data_validation],
    wait=True
)

training_job_name = model_trainer.base_job_name
print(f'\n Training completed: {training_job_name}')

### MLflow에서 훈련 결과 보기

훈련 결과를 보려면:

1. SageMaker Studio에서 왼쪽 사이드바의 **MLflow**를 클릭합니다
2. MLflow 앱(예: DefaultMLFlowApp)을 찾습니다
3. 클릭하여 MLflow UI를 엽니다
4. 실험으로 이동합니다: **demo-ml-xgb-train-monitor**
5. 훈련 중 로깅된 파라미터, 메트릭, 아티팩트를 확인합니다

---

## 5: 데이터 캡처를 통한 실시간 엔드포인트

데이터 캡처가 활성화된 SageMaker 실시간 엔드포인트로 훈련된 모델을 배포합니다. 데이터 캡처는 엔드포인트에 대한 입력과 배포된 모델의 추론 출력을 Amazon S3에 로깅합니다.

### 데이터 캡처를 통한 실시간 엔드포인트를 사용하는 이유?
- **낮은 지연 시간**: 실시간 애플리케이션을 위한 1초 미만의 추론
- **데이터 캡처**: 요청/응답 페이로드를 S3에 자동으로 로깅
- **모니터링**: 캡처된 데이터로 드리프트 감지 및 모델 품질 모니터링 가능
- **프로덕션 준비**: 자동 스케일링, A/B 테스트, 섀도우 배포 지원

### MLflow 모델 레지스트리를 사용한 ModelBuilder로 배포

`InferenceSpec`과 함께 `ModelBuilder`를 사용하여 MLflow에 등록된 모델을 직접 배포합니다 — 모델 재패키징이 필요 없습니다. `InferenceSpec`은 MLflow에서 모델을 로드하고 추론을 실행하는 방법을 정의합니다. 이는 [노트북 02](02-sagemaker-containers.ipynb)에서 사용된 것과 동일한 패턴입니다.

In [ ]:
from sagemaker.serve.mode.function_pointers import Mode
from sagemaker.serve.spec.inference_spec import InferenceSpec
from sagemaker.serve.utils.types import ModelServer
from mlflow import MlflowClient

class XGBoostMLflowSpec(InferenceSpec):
    def __init__(self, mlflow_model_uri, tracking_uri):
        self._mlflow_model_uri = mlflow_model_uri
        self._tracking_uri = tracking_uri

    def load(self, model_dir=None):
        import mlflow
        mlflow.set_tracking_uri(self._tracking_uri)
        return mlflow.xgboost.load_model(self._mlflow_model_uri)

    def invoke(self, input_object, model):
        import xgboost as xgb
        import numpy as np
        if isinstance(input_object, str):
            rows = [list(map(float, r.split(","))) for r in input_object.strip().split("\n") if r]
            dmatrix = xgb.DMatrix(np.array(rows), feature_names=model.feature_names)
        else:
            dmatrix = xgb.DMatrix(np.array(input_object), feature_names=model.feature_names)
        return "\n".join(map(str, model.predict(dmatrix)))


In [ ]:
# Get the latest version of the registered model from MLflow
client = MlflowClient()
registered_model = client.get_registered_model(name=mlflow_model_name)
latest_version = registered_model.latest_versions[0]
# Use models:/name/version format (MLflow 3.x native source URI is not compatible with ModelBuilder)
mlflow_model_path = f"models:/{registered_model.name}/{latest_version.version}"
print(f'MLflow model path: {mlflow_model_path}')


`ModelBuilder`를 사용하여 `InferenceSpec`으로 모델을 빌드합니다. 이를 통해 모델 아티팩트를 재패키징할 필요가 없습니다 — 모델은 추론 시 MLflow에서 직접 로드됩니다.

In [ ]:
from pydantic import ValidationError
from sagemaker.core.model_monitor import DataCaptureConfig

model_builder = ModelBuilder(
    image_uri=xgboost_image,
    instance_type='ml.m5.xlarge',
    inference_spec=XGBoostMLflowSpec(mlflow_model_path, mlflow_arn),
    schema_builder=SchemaBuilder(
        sample_input='1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20',
        sample_output='0.7'
    ),
    model_server=ModelServer.TORCHSERVE,
    dependencies={'auto': False},
)

built_model = model_builder.build()
model_name = model_builder.model_name
print(f'Model built: {model_name}')


ModelBuilder로 생성된 모델 객체는 실시간 엔드포인트, 비동기 엔드포인트, 서버리스 추론, 배치 변환을 사용하여 여러 번 배포할 수 있습니다. 이 경우 모델은 데이터 캡처가 활성화된 실시간 엔드포인트로 배포됩니다.

In [ ]:
%%time
capture_s3_uri = f's3://{bucket_name}/{bucket_prefix}/data-capture'

data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=capture_s3_uri,
    capture_options=['Input', 'Output'],
    csv_content_types=['text/csv'],
)

try:
    endpoint = model_builder.deploy(
        instance_type='ml.m5.xlarge',
        initial_instance_count=1,
        data_capture_config=data_capture_config,
    )
except ValidationError:
    # SDK bug: Endpoint.get() fails deserializing data_capture_config when kms_key_id is absent.
    # The endpoint is already deployed successfully at this point.
    pass

endpoint_name = model_builder.endpoint_name
print(f'Endpoint deployed: {endpoint_name}')
print(f'Data capture destination: {capture_s3_uri}')

### 실측 데이터 준비 및 엔드포인트 호출

모델 품질(정확도, F1 점수)을 계산하려면 예측 레이블과 비교할 실측 레이블이 있어야 합니다. 프로덕션에서 실측 레이블은 일반적으로 비동기적으로 수집됩니다. 이 샘플 노트북에서는 테스트 데이터셋을 사용하여 모델 품질을 계산하므로 실측 레이블이 이미 있습니다. 각 추론 입력에는 나중에 병합을 위한 고유 ID(예: `inf-000010`)가 할당됩니다.

In [ ]:
test_features = X_test.copy()

# SIMULATED PRODUCTION DRIFT: apply shifts here so the Data Capture actually
# contains drifted records (the Processing Job compares capture vs baseline).
# Do NOT do this in production — this is purely to demonstrate drift detection.
if 'age' in test_features.columns:
    test_features['age'] = test_features['age'] + 2

if 'duration' in test_features.columns:
    test_features['duration'] = test_features['duration'] * 1.15

if 'campaign' in test_features.columns:
    test_features['campaign'] = test_features['campaign'] + np.random.normal(0, 0.2, len(test_features))

test_features.to_csv('data/test_features.csv', index=False, header=False)

ground_truth = {}
for idx, (i, row) in enumerate(test_features.iterrows()):
    inference_id = f'inf-{idx:06d}'
    ground_truth[inference_id] = int(y_test.iloc[idx])

gt_df = pd.DataFrame(list(ground_truth.items()), columns=['inference_id', 'target'])
gt_df.to_csv('data/ground_truth.csv', index=False)
print(f'Ground truth stored for {len(gt_df)} records')


이제 실측 레이블 없이 테스트 데이터셋을 실시간 엔드포인트에 추론을 위해 전송할 수 있습니다.

In [ ]:
runtime_client = boto3.client('sagemaker-runtime', region_name=region)

print(f'Sending {len(test_features)} requests to endpoint...')
for idx, (i, row) in enumerate(test_features.iterrows()):
    inference_id = f'inf-{idx:06d}'
    payload = ','.join(str(v) for v in row.values)
    response = runtime_client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='text/csv',
        Accept='text/csv',
        Body=payload,
        InferenceId=inference_id
    )
    response['Body'].read()  # drain the response

print(f'Sent {len(test_features)} requests')

데이터 캡처 결과가 S3에 도착하는 데 몇 분이 걸릴 수 있습니다. 데이터 캡처는 추론 요청에 영향을 미치지 않도록 높은 디스크 사용량에서 요청 캡처를 중지합니다.

In [ ]:
import time as _time

s3_client = boto3.client('s3')
capture_prefix = f'{bucket_prefix}/data-capture/{endpoint_name}/AllTraffic'

print('Waiting for data capture files in S3 (may take 1-2 minutes)...')
for attempt in range(12):
    resp = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=capture_prefix, MaxKeys=1)
    if resp.get('KeyCount', 0) > 0:
        print(f'Data capture files found after {(attempt+1)*15}s')
        break
    _time.sleep(15)
else:
    print('No capture files found yet. They may still be buffering.')

# List captured files
resp = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=capture_prefix)
capture_keys = [obj['Key'] for obj in resp.get('Contents', [])]
print(f'\nFound {len(capture_keys)} capture file(s)')

데이터 캡처는 입력과 출력을 JSONL 파일에 저장합니다. 이 파일은 나중에 드리프트를 계산하기 위해 pandas DataFrame에 로드할 수 있습니다.

In [ ]:
import base64

captured_records = []

for key in capture_keys:
    obj = s3_client.get_object(Bucket=bucket_name, Key=key)
    for line in obj['Body'].read().decode('utf-8').strip().split('\n'):
        record = json.loads(line)
        inference_id = record.get('eventMetadata', {}).get('inferenceId', '')

        input_data = record.get('captureData', {}).get('endpointInput', {})
        output_data = record.get('captureData', {}).get('endpointOutput', {})

        raw_input = input_data.get('data', '')
        if input_data.get('encoding') == 'BASE64':
            raw_input = base64.b64decode(raw_input).decode('utf-8')

        raw_output = output_data.get('data', '')
        if output_data.get('encoding') == 'BASE64':
            raw_output = base64.b64decode(raw_output).decode('utf-8')

        captured_records.append({
            'inference_id': inference_id,
            'input': raw_input.strip(),
            'prediction_proba': float(raw_output.strip()) if raw_output.strip() else None
        })

captured_df = pd.DataFrame(captured_records)
captured_df['prediction'] = (captured_df['prediction_proba'] > 0.5).astype(int)

feature_cols = pd.DataFrame(
    captured_df['input'].str.split(',').tolist(),
    columns=test_features.columns
).astype(float)
captured_df = pd.concat([captured_df[['inference_id', 'prediction_proba', 'prediction']], feature_cols], axis=1)

print(f'Parsed {len(captured_df)} captured records with inferenceId')
print(captured_df[['inference_id', 'prediction_proba', 'prediction']].head())

---

## 6: Evidently를 사용한 데이터 드리프트 모니터링

### 데이터 드리프트란?

데이터 드리프트는 입력 피처의 통계적 특성이 시간이 지남에 따라 변화할 때 발생하며, 이는 모델 성능을 저하시킬 수 있습니다.

### 드리프트를 모니터링하는 이유?
- **조기 경고**: 비즈니스에 영향을 미치기 전에 문제 감지
- **모델 노후화**: 모델 재훈련이 필요한 시점 파악
- **데이터 품질**: 데이터 파이프라인 문제 식별
- **규정 준수**: 규제 요구사항을 위한 모델 성능 추적

### Evidently 기능
- **오픈소스**: 무료이며 커스터마이즈 가능
- **통계 테스트**: 다양한 드리프트 감지 방법 (KS 테스트, 카이제곱 등)
- **시각적 보고서**: 인터랙티브 HTML 보고서
- **유연성**: 모든 ML 프레임워크와 호환
- **MLflow 통합**: 메트릭과 아티팩트를 MLflow에 로깅

데이터 드리프트를 계산하려면 먼저 모델 훈련 중 저장된 기준 데이터셋을 로드합니다.

In [ ]:
reference_data = pd.read_csv('data/baseline.csv')

print(f'Reference (baseline) data shape: {reference_data.shape}')
print('\nReference data represents the training data distribution')
print('\nReference data statistics:')
print(reference_data.describe())

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
    <h3>⚠️ 인위적 드리프트가 포함된 시뮬레이션된 프로덕션 데이터</h3>
<p><strong>목적:</strong> 드리프트 감지 기능을 시연하기 위해 제어된 데이터 드리프트를 도입합니다</p>

<b>중요:</b> 데모 목적으로만 테스트 세트에 인위적인 데이터 드리프트를 도입합니다. 프로덕션에서는 사용하지 마세요. 실제 시나리오에서는 실제 프로덕션 데이터가 됩니다.

In [ ]:
feature_columns = [c for c in captured_df.columns if c not in ['inference_id', 'prediction_proba', 'prediction']]
production_data = captured_df[feature_columns].copy()

print(f'Production data shape: {production_data.shape}')
print('\nProduction data statistics:')
print(production_data.describe())
print('\nDrift was already injected into the payloads sent to the endpoint (see Section 5),')
print('so the Data Capture files — and therefore this DataFrame — already contain the shifted distribution.')


Evidently는 다양한 유형의 [데이터 드리프트](https://docs.evidentlyai.com/metrics/preset_data_drift)를 계산하기 위한 다양한 프리셋을 제공하며, 모두 특정 ML 사용 사례에 맞게 커스터마이즈할 수 있습니다. MLflow에 최종 보고서를 저장하는 것 외에도 특정 드리프트 값을 추출하여 MLflow에 별도의 메트릭으로 저장하는 것이 유용합니다. 아래 셀에는 MLflow에 로깅하기 위해 Evidently 결과에서 특정 메트릭을 추출하는 헬퍼 함수가 포함되어 있습니다.

In [ ]:
import numpy as np

def log_metrics_from_dict(d):
    metrics = d.get("metrics", []) or []

    drifted_columns_logged = False

    for m in metrics:
        metric_name = m.get("metric_name", "")
        cfg = m.get("config", {}) or {}
        val = m.get("value")

        # 1) DriftedColumnsCount: always log
        if metric_name.startswith("DriftedColumnsCount"):
            drifted_columns_logged = True
            # value is a dict: {"count": ..., "share": ...}
            if isinstance(val, dict):
                count = float(val.get("count", 0.0))
                share = float(val.get("share", 0.0))
            else:
                # Fallback if structure changes
                count = float(val) if val is not None else 0.0
                share = 0.0

            mlflow.log_metric("DriftedColumnsCount.count", count)
            mlflow.log_metric("DriftedColumnsCount.share", share)
            continue

        # 2) ValueDrift: log only if value > threshold
        if metric_name.startswith("ValueDrift"):
            threshold = cfg.get("threshold")
            column = cfg.get("column")

            if threshold is None or column is None:
                continue

            try:
                numeric_val = float(val)
            except (TypeError, ValueError):
                continue

            if numeric_val > float(threshold):
                key = f"ValueDrift:{column}"
                mlflow.log_metric(key, numeric_val)

    # 3) If there was no DriftedColumnsCount metric at all, log 0
    if not drifted_columns_logged:
        mlflow.log_metric("DriftedColumnsCount", 0.0)

특정 메트릭을 추출하는 것 외에도 Evidently가 생성한 전체 보고서가 별도의 폴더에 저장됩니다.

In [ ]:
os.makedirs('reports', exist_ok=True)

이 샘플은 Evidently의 내장 `DataDriftPreset`과 `DataSummaryPreset`을 사용하여 기준 데이터와 추론 결과에서 데이터 드리프트를 계산합니다. 메트릭이 추출되고 HTML 및 JSON 보고서와 함께 SageMaker MLflow 앱에 저장됩니다.

In [ ]:
mlflow.set_experiment(mlflow_experiment_name)
timestamp_suffix = datetime.now().strftime('%Y%m%d_%H%M%S')

with mlflow.start_run(run_name=f"data_drift_quality_monitoring_{timestamp_suffix}") as drift_run:
    drift_run_id = drift_run.info.run_id
    
    mlflow.log_params({
        'reference_data_size': len(reference_data),
        'current_data_size': len(production_data),
        'monitoring_timestamp': datetime.now().isoformat(),
        'model_name': model_name,
        'training_job': training_job_name
    })

    # Wrap DataFrames in Evidently Dataset objects
    drift_data_definition = DataDefinition()
    reference_dataset = Dataset.from_pandas(reference_data, data_definition=drift_data_definition)
    production_dataset = Dataset.from_pandas(production_data, data_definition=drift_data_definition)
    
    print('Creating Evidently data drift report...')
    data_drift_report = Report(metrics=[
        DataDriftPreset(),
    ])
    data_drift_report_snapshot = data_drift_report.run(
        reference_data=reference_dataset,
        current_data=production_dataset
    )
    
    data_drift_report_filename = f'data_drift_report_{timestamp_suffix}'
    data_drift_report_snapshot.save_html(f'reports/{data_drift_report_filename}.html')
    data_drift_report_snapshot.save_json(f'reports/{data_drift_report_filename}.json')
    
    mlflow.log_artifact(f'reports/{data_drift_report_filename}.html', 'evidently_report_data_drift_html')
    mlflow.log_artifact(f'reports/{data_drift_report_filename}.json', 'evidently_report_data_drift_json')
    
    drift_results = data_drift_report_snapshot.dict()
    log_metrics_from_dict(drift_results)

    print('Creating Evidently data summary report...')
    data_quality_report = Report(metrics=[
        DataSummaryPreset(),
    ])
    data_quality_report_snapshot = data_quality_report.run(
        reference_data=reference_data,
        current_data=production_data
    )
    
    data_quality_filename_html = f'data_quality_report_{timestamp_suffix}.html'
    data_quality_report_snapshot.save_html(f'reports/{data_quality_filename_html}')
    print(f'Data quality report saved to: {data_quality_filename_html}')
    mlflow.log_artifact(f'reports/{data_quality_filename_html}', 'evidently_report_data_quality')

Evidently가 생성한 최종 보고서는 이 노트북 내에서 직접 볼 수도 있습니다.

In [ ]:
from IPython.display import IFrame

print('Displaying Evidently Data Drift Report:')
IFrame(src=f'reports/{data_drift_report_filename}.html', width=1000, height=800)

### MLflow에서 드리프트 메트릭 보기

MLflow에서 드리프트 모니터링 결과를 보려면:

1. SageMaker Studio MLflow UI를 엽니다
2. 실험으로 이동합니다: **demo-ml-xgb-train-monitor**
3. **data_drift_**로 시작하는 이름의 실행을 찾습니다
4. 다음을 확인합니다:
   - **메트릭**: 각 피처의 드리프트 점수, 데이터셋 수준 드리프트
   - **아티팩트**: HTML 보고서 및 JSON 요약
   - **파라미터**: 모니터링 설정 및 타임스탬프

---

## 7: Evidently를 사용한 이진 분류 모델 품질 평가

분류 평가는 예측 레이블을 실제 실측 레이블과 비교하여 모델이 예측 작업을 얼마나 잘 수행하는지 평가합니다.

### 주요 분류 메트릭

- **정확도**: 예측의 전반적인 정확성
- **정밀도**: 모든 양성 예측 중 실제로 양성인 비율 (거짓 양성 최소화)
- **재현율**: 모든 실제 양성 중 올바르게 예측된 비율 (거짓 음성 최소화)
- **F1 점수**: 정밀도와 재현율의 조화 평균 (균형 메트릭)
- **ROC-AUC**: ROC 곡선 아래 면적 (판별 능력 측정)
- **혼동 행렬**: 진양성, 진음성, 거짓양성, 거짓음성의 분류

먼저 실시간 엔드포인트가 생성한 예측과 이전에 저장된 실측 레이블을 병합합니다. 이 두 데이터셋은 고유 ID로 병합할 수 있습니다.

In [ ]:
gt_df = pd.read_csv('data/ground_truth.csv')
classification_eval_data = captured_df.merge(gt_df, on='inference_id', how='inner')

print(f'Merged dataset shape: {classification_eval_data.shape}')
print(f'  Captured records: {len(captured_df)}')
print(f'  Ground truth records: {len(gt_df)}')
print(f'  Matched records: {len(classification_eval_data)}')
print(f'\nActual labels distribution:')
print(classification_eval_data['target'].value_counts())
print(f'\nPredicted labels distribution:')
print(classification_eval_data['prediction'].value_counts())

데이터 드리프트와 유사하게, Evidently는 다양한 유형의 [모델 품질](https://docs.evidentlyai.com/metrics/preset_classification)을 계산하기 위한 다양한 프리셋을 제공합니다. 아래 셀에는 MLflow에 로깅하기 위해 Evidently 결과에서 특정 메트릭을 추출하는 헬퍼 함수가 포함되어 있습니다.

In [ ]:
import numpy as np

def log_evidently_classification_metrics_to_mlflow(metrics_dict):
    """Log Evidently classification metrics to MLflow using only metric prefix"""
    for metric in metrics_dict.get('metrics', []):
        # Extract only prefix before first '('
        metric_name_full = metric.get('metric_name', '')
        metric_name = metric_name_full.split('(')[0].strip()
        
        value = metric.get('value')
        
        if isinstance(value, dict):
            # Handle dict values like F1ByLabel, PrecisionByLabel, etc.
            for label, val in value.items():
                mlflow.log_metric(f"{metric_name}_label_{label}", float(val))
        else:
            # Handle scalar values (including np.float64)
            mlflow.log_metric(metric_name, float(value))

이 샘플은 Evidently의 내장 `ClassificationPreset`을 사용하여 추론 결과와 실측 레이블에서 모델 품질을 계산합니다. 메트릭이 추출되고 HTML 및 JSON 보고서와 함께 SageMaker MLflow 앱에 저장됩니다.

In [ ]:
with mlflow.start_run(run_name=f'model_quality_{timestamp_suffix}') as run:
    eval_data = classification_eval_data.copy()
    print(f"Evaluation dataset columns: {eval_data.columns.tolist()}")
    print(f"Evaluation dataset shape: {eval_data.shape}")
    
    # Create DataDefinition with BinaryClassification to tell Evidently which columns to use
    # This is REQUIRED for ClassificationPreset to work
    data_definition = DataDefinition(
        classification=[
            BinaryClassification(
                target='target',              # Column with actual labels
                prediction_labels='prediction', # Column with predicted labels
                pos_label=1,                  # Positive class label
            )
        ],
    )
    
    # Wrap the DataFrame in an Evidently Dataset
    eval_dataset = Dataset.from_pandas(
        eval_data,
        data_definition=data_definition,
    )
    
    print('\nCreating Evidently classification performance report...')
    classification_report = Report(metrics=[
        ClassificationPreset(),
    ])
    classification_report_snapshot = classification_report.run(
        reference_data=None,
        current_data=eval_dataset,
    )
    
    classification_report_filename = f'classification_report_{timestamp_suffix}'
    classification_report_snapshot.save_html(f'data/{classification_report_filename}.html')
    print(f'Classification report saved to: data/{classification_report_filename}.html')
    
    classification_report_snapshot.save_json(f'data/{classification_report_filename}.json')
    print(f'Classification report saved to: data/{classification_report_filename}.json')
    
    mlflow.log_artifact(f'data/{classification_report_filename}.html', 'evidently_classification_report_html')
    mlflow.log_artifact(f'data/{classification_report_filename}.json', 'evidently_classification_report_json')
    
    classification_report_dict = classification_report_snapshot.dict()
    log_evidently_classification_metrics_to_mlflow(classification_report_dict)

### 분류 평가 요약

분류 보고서는 다음을 제공합니다:

1. **전반적인 성능 메트릭**
   - 정확도: 올바른 예측의 비율
   - 정밀도: 양성 예측의 품질
   - 재현율: 실제 양성 케이스의 커버리지
   - F1 점수: 정밀도와 재현율 간의 균형
   - ROC-AUC: 모든 임계값에서의 판별 능력

2. **혼동 행렬**
   - 진양성 (TP): 올바르게 예측된 양성 케이스
   - 진음성 (TN): 올바르게 예측된 음성 케이스
   - 거짓양성 (FP): 양성으로 잘못 예측 (1종 오류)
   - 거짓음성 (FN): 음성으로 잘못 예측 (2종 오류)

3. **클래스 수준 분석**
   - 클래스별 성능 분류 (0: 미가입, 1: 가입)
   - 지원: 클래스당 샘플 수
   - 클래스별 정밀도 및 재현율

### 비즈니스 영향

은행 마케팅 사용 사례의 경우:
- **높은 정밀도**: 가입 가능성이 낮은 고객에 대한 낭비되는 노력 최소화
- **높은 재현율**: 잠재적 가입자 포착 극대화
- **F1 점수**: 최적의 캠페인 효율성을 위해 두 목표 균형
- **임계값 조정**: 비즈니스 비용/이익에 따라 결정 경계 조정

---

## 8: 예약된 파이프라인에서 모니터링 자동화

위의 Evidently 계산은 임시 분석에 적합하지만, 프로덕션에서는 노트북을 열지 않고 드리프트 및 모델 품질 검사가 일정에 따라 자동으로 실행되기를 원합니다. Evidently 로직을 **SageMaker 파이프라인**으로 오케스트레이션되는 **SageMaker 처리 작업**으로 래핑합니다:

- 엔드포인트에서 최신 데이터 캡처 파일을 가져와 훈련 기준과 비교
- Evidently 보고서와 드리프트 메트릭을 MLflow에 로깅
- 드리프트된 피처의 비율이 임계값을 초과하면 **SNS** 알림 게시

![실시간 엔드포인트를 사용한 모델 모니터링 워크플로우](img/arch-sagemaker-inference-predictiveml-monitoring-RealTime-inf.png "실시간 엔드포인트를 사용한 모델 모니터링 워크플로우")

### 드리프트 알림을 위한 SNS 주제 생성

아래 셀은 SNS 주제를 생성하고 이메일 주소를 구독합니다. 이메일 구독은 이메일 소유자의 확인이 필요합니다. 실행 역할에 SNS 권한이 없으면 셀이 경고를 로깅하고 파이프라인은 계속 실행됩니다 — 알림만 게시되지 않습니다.

In [ ]:
ALERT_EMAIL = ''  # optional: set to receive drift alerts by email (must be confirmed)
SNS_TOPIC_NAME = 'sagemaker-evidently-drift-alerts'

sns_topic_arn = ''
try:
    sns_client = boto3.client('sns', region_name=region)
    sns_topic_arn = sns_client.create_topic(Name=SNS_TOPIC_NAME)['TopicArn']
    print(f'SNS topic ready: {sns_topic_arn}')
    if ALERT_EMAIL:
        sns_client.subscribe(TopicArn=sns_topic_arn, Protocol='email', Endpoint=ALERT_EMAIL)
        print(f'Subscription requested for {ALERT_EMAIL} — confirm via email to activate.')
except Exception as e:
    print(f'[WARN] Could not set up SNS ({e}). Pipeline will still run without alerts.')


### 처리 작업 스크립트 작성

SageMaker 처리 작업이 참조할 수 있도록 Evidently 모니터링 스크립트를 `scripts/evidently_monitor.py`에 구체화합니다.

In [ ]:
%%writefile scripts/evidently_monitor.py
import argparse
import base64
import glob
import json
import os
from datetime import datetime, timedelta, timezone

import boto3
import mlflow
import pandas as pd
from evidently import BinaryClassification, Dataset, DataDefinition, Report
from evidently.presets import ClassificationPreset, DataDriftPreset, DataSummaryPreset


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--mlflow-tracking-uri", required=True)
    p.add_argument("--mlflow-experiment", required=True)
    p.add_argument("--drift-threshold", type=float, default=0.05)
    p.add_argument("--lookback-days", type=int, default=7,
                   help="Only consider capture records within the last N days. 0 disables the filter.")
    p.add_argument("--critical-features", default="",
                   help="Comma-separated features that alert individually if drifted (e.g. 'duration,age').")
    p.add_argument("--min-f1", type=float, default=0.0,
                   help="If >0, alert when ClassificationPreset F1 falls below this value.")
    p.add_argument("--sns-topic-arn", default="")
    p.add_argument("--endpoint-name", default="")
    return p.parse_args()


def load_capture(capture_dir, feature_names, lookback_days):
    cutoff = None
    if lookback_days and lookback_days > 0:
        cutoff = datetime.now(timezone.utc) - timedelta(days=lookback_days)
    rows, skipped = [], 0
    for path in glob.glob(os.path.join(capture_dir, "**", "*.jsonl"), recursive=True):
        with open(path) as f:
            for line in f:
                rec = json.loads(line)
                meta = rec.get("eventMetadata", {})
                if cutoff is not None:
                    ts_str = meta.get("inferenceTime", "")
                    try:
                        ts = datetime.fromisoformat(ts_str.replace("Z", "+00:00"))
                    except Exception:
                        ts = None
                    if ts is not None and ts < cutoff:
                        skipped += 1
                        continue
                cap = rec.get("captureData", {})
                inp, out = cap.get("endpointInput", {}), cap.get("endpointOutput", {})
                raw_in = inp.get("data", "")
                if inp.get("encoding") == "BASE64":
                    raw_in = base64.b64decode(raw_in).decode("utf-8")
                raw_out = out.get("data", "")
                if out.get("encoding") == "BASE64":
                    raw_out = base64.b64decode(raw_out).decode("utf-8")
                rows.append({
                    "inference_id": meta.get("inferenceId", ""),
                    "input": raw_in.strip(),
                    "proba": float(raw_out.strip()) if raw_out.strip() else None,
                })
    if skipped:
        print(f"Skipped {skipped} capture records older than {lookback_days} day(s).")
    if not rows:
        return None
    df = pd.DataFrame(rows)
    features = pd.DataFrame(df["input"].str.split(",").tolist(), columns=feature_names).astype(float)
    df["prediction"] = (df["proba"] > 0.5).astype(int)
    return pd.concat([df[["inference_id", "proba", "prediction"]], features], axis=1)


def drifted_share(report_dict):
    for m in report_dict.get("metrics", []):
        if m.get("metric_name", "").startswith("DriftedColumnsCount"):
            v = m.get("value")
            return float(v.get("share", 0.0)) if isinstance(v, dict) else 0.0
    return 0.0


def drifted_critical(report_dict, critical):
    """Return list of critical features whose ValueDrift exceeded their threshold."""
    hits = []
    if not critical:
        return hits
    for m in report_dict.get("metrics", []):
        if not m.get("metric_name", "").startswith("ValueDrift"):
            continue
        cfg = m.get("config", {}) or {}
        column, threshold, val = cfg.get("column"), cfg.get("threshold"), m.get("value")
        if column in critical and threshold is not None and val is not None:
            try:
                if float(val) > float(threshold):
                    hits.append(column)
            except (TypeError, ValueError):
                pass
    return hits


def all_drifted(report_dict):
    """Return list of (column, value) for every column whose ValueDrift exceeded its threshold, sorted desc."""
    hits = []
    for m in report_dict.get("metrics", []):
        if not m.get("metric_name", "").startswith("ValueDrift"):
            continue
        cfg = m.get("config", {}) or {}
        column, threshold, val = cfg.get("column"), cfg.get("threshold"), m.get("value")
        if column and threshold is not None and val is not None:
            try:
                fv = float(val)
                if fv > float(threshold):
                    hits.append((column, fv))
            except (TypeError, ValueError):
                pass
    return sorted(hits, key=lambda x: x[1], reverse=True)


def classification_f1(report_dict):
    for m in report_dict.get("metrics", []):
        if m.get("metric_name", "").startswith("F1Score") and not m.get("metric_name", "").startswith("F1ByLabel"):
            try:
                return float(m.get("value"))
            except (TypeError, ValueError):
                return None
    return None


def publish_alert(topic_arn, subject, message):
    if not topic_arn:
        return
    try:
        # Extract region from the SNS topic ARN (arn:aws:sns:<region>:<acct>:<name>)
        region = topic_arn.split(":")[3] if topic_arn.count(":") >= 3 else os.environ.get("AWS_REGION", "us-east-1")
        boto3.client("sns", region_name=region).publish(TopicArn=topic_arn, Subject=subject, Message=message)
        print(f"SNS alert published: {subject}")
    except Exception as e:
        print(f"[WARN] SNS publish skipped: {e}")


def main():
    args = parse_args()
    base_in, base_out = "/opt/ml/processing/input", "/opt/ml/processing/output"
    os.makedirs(base_out, exist_ok=True)

    reference = pd.read_csv(f"{base_in}/baseline/baseline.csv")
    captured = load_capture(f"{base_in}/capture", feature_names=reference.columns.tolist(),
                            lookback_days=args.lookback_days)
    if captured is None or len(captured) == 0:
        print(f"[WARN] No capture records found within the last {args.lookback_days} day(s). "
              f"Nothing to compare — exiting successfully.")
        return

    production = captured[reference.columns.tolist()].copy()
    critical = [c.strip() for c in args.critical_features.split(",") if c.strip()]

    mlflow.set_tracking_uri(args.mlflow_tracking_uri)
    mlflow.set_experiment(args.mlflow_experiment)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    with mlflow.start_run(run_name=f"scheduled_monitoring_{ts}"):
        mlflow.log_params({
            "reference_size": len(reference),
            "current_size": len(production),
            "drift_threshold": args.drift_threshold,
            "lookback_days": args.lookback_days,
            "critical_features": args.critical_features,
            "min_f1": args.min_f1,
            "endpoint_name": args.endpoint_name,
        })

        snap = Report(metrics=[DataDriftPreset(), DataSummaryPreset()]).run(
            reference_data=Dataset.from_pandas(reference, data_definition=DataDefinition()),
            current_data=Dataset.from_pandas(production, data_definition=DataDefinition()),
        )
        html_path, json_path = f"{base_out}/data_drift_{ts}.html", f"{base_out}/data_drift_{ts}.json"
        snap.save_html(html_path)
        snap.save_json(json_path)
        mlflow.log_artifact(html_path, "evidently_report_data_drift_html")
        mlflow.log_artifact(json_path, "evidently_report_data_drift_json")

        drift_dict = snap.dict()
        share = drifted_share(drift_dict)
        mlflow.log_metric("DriftedColumnsCount.share", share)
        print(f"Drifted columns share: {share:.3f} (threshold={args.drift_threshold})")

        all_hits = all_drifted(drift_dict)
        if all_hits:
            print(f"All drifted features: {all_hits}")

        critical_hits = drifted_critical(drift_dict, critical)
        if critical_hits:
            print(f"Critical features drifted: {critical_hits}")

        f1 = None
        gt_path = f"{base_in}/ground_truth/ground_truth.csv"
        if os.path.exists(gt_path):
            gt = pd.read_csv(gt_path)
            eval_df = captured.merge(gt, on="inference_id", how="inner")
            if len(eval_df):
                dd = DataDefinition(classification=[
                    BinaryClassification(target="target", prediction_labels="prediction", pos_label=1)
                ])
                cls_snap = Report(metrics=[ClassificationPreset()]).run(
                    reference_data=None,
                    current_data=Dataset.from_pandas(eval_df, data_definition=dd),
                )
                cls_html = f"{base_out}/classification_{ts}.html"
                cls_snap.save_html(cls_html)
                mlflow.log_artifact(cls_html, "evidently_classification_report_html")
                f1 = classification_f1(cls_snap.dict())
                if f1 is not None:
                    mlflow.log_metric("F1Score", f1)
                    print(f"Classification F1: {f1:.3f} (min_f1={args.min_f1})")

        # Consolidate triggers
        reasons = []
        if share > args.drift_threshold:
            reasons.append(f"drifted columns share {share:.2%} > threshold {args.drift_threshold:.2%}")
        if critical_hits:
            reasons.append(f"critical features drifted: {', '.join(critical_hits)}")
        if args.min_f1 > 0 and f1 is not None and f1 < args.min_f1:
            reasons.append(f"F1 {f1:.3f} < min_f1 {args.min_f1:.3f}")

        if reasons:
            run = mlflow.active_run()
            run_id = run.info.run_id if run else "n/a"
            experiment_id = run.info.experiment_id if run else "n/a"
            mlflow_ui_hint = (
                f"Open SageMaker Studio -> MLflow -> Experiment '{args.mlflow_experiment}' "
                f"-> run_id {run_id}"
            )
            drifted_block = ""
            if all_hits:
                drifted_block = "Drifted features (feature: score):\n  - " + "\n  - ".join(
                    f"{col}: {score:.4f}" for col, score in all_hits
                ) + "\n\n"

            publish_alert(
                args.sns_topic_arn,
                subject=f"[SageMaker] Model monitoring alert — {args.endpoint_name}",
                message=(
                    f"Endpoint: {args.endpoint_name}\n"
                    f"Run timestamp: {ts}\n\n"
                    f"Triggers:\n  - " + "\n  - ".join(reasons) + "\n\n"
                    + drifted_block +
                    f"MLflow experiment: {args.mlflow_experiment} (id {experiment_id})\n"
                    f"MLflow run_id:     {run_id}\n"
                    f"{mlflow_ui_hint}\n"
                ),
            )


if __name__ == "__main__":
    main()


### 모니터링 파이프라인 정의

`FrameworkProcessor`에서 `scripts/evidently_monitor.py`를 실행하는 단일 `ProcessingStep`. 파이프라인 파라미터를 통해 재배포 없이 각 실행을 조정할 수 있습니다:

- `DriftThreshold` — SNS 알림을 트리거하는 드리프트된 컬럼의 비율
- `LookbackDays` — 최근 N일 이내의 캡처 레코드만 고려 (비활성화하려면 `0`으로 설정)
- `CriticalFeatures` — 전체 비율이 임계값 미만이더라도 드리프트 시 개별적으로 알림을 발생시키는 쉼표로 구분된 피처
- `MinF1` — > 0이면 분류 F1이 이 값 미만으로 떨어질 때도 알림 (실측값 필요)
- `SnsTopicArn`, `EndpointName` — 앞서 생성된 SNS 주제와 엔드포인트에 연결

SageMaker 관리형 scikit-learn 이미지에는 Evidently나 MLflow가 포함되어 있지 않습니다. 스크립트 옆의 `requirements.txt`는 처리 스크립트를 실행하기 전에 `FrameworkProcessor`에 의해 자동으로 설치됩니다.

In [ ]:
%%writefile scripts/requirements.txt
evidently==0.7.17
mlflow<4
sagemaker-mlflow


In [ ]:
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.mlops.workflow.steps import ProcessingStep, CacheConfig
from sagemaker.core.workflow.parameters import ParameterFloat, ParameterString, ParameterInteger
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.core.processing import FrameworkProcessor
from sagemaker.core.shapes import ProcessingInput, ProcessingS3Input, ProcessingOutput, ProcessingS3Output
from sagemaker.core import image_uris

monitoring_pipeline_name = 'evidently-monitoring-pipeline'

p_drift_threshold = ParameterFloat(name='DriftThreshold', default_value=0.05)
p_sns_topic = ParameterString(name='SnsTopicArn', default_value=sns_topic_arn or '')
p_endpoint = ParameterString(name='EndpointName', default_value=endpoint_name)
p_lookback_days = ParameterInteger(name='LookbackDays', default_value=7)
p_critical_features = ParameterString(name='CriticalFeatures', default_value='duration,age')
p_min_f1 = ParameterFloat(name='MinF1', default_value=0.0)

endpoint_capture_s3 = f'{capture_s3_uri}/{endpoint_name}/AllTraffic'

ground_truth_s3 = sagemaker_session.upload_data(
    'data/ground_truth.csv', bucket_name, f'{bucket_prefix}/data/ground-truth'
)

processor_image_uri = image_uris.retrieve(framework='sklearn', version='1.4-2', region=region)

monitor_processor = FrameworkProcessor(
    image_uri=processor_image_uri,
    role=sm_role,
    instance_type='ml.m5.large',
    instance_count=1,
    sagemaker_session=PipelineSession(),
    base_job_name='evidently-monitor',
    command=['python3'],
)

monitor_inputs = [
    ProcessingInput(input_name='baseline', s3_input=ProcessingS3Input(
        s3_uri=baseline_s3, local_path='/opt/ml/processing/input/baseline', s3_data_type='S3Prefix')),
    ProcessingInput(input_name='capture', s3_input=ProcessingS3Input(
        s3_uri=endpoint_capture_s3, local_path='/opt/ml/processing/input/capture', s3_data_type='S3Prefix')),
    ProcessingInput(input_name='ground_truth', s3_input=ProcessingS3Input(
        s3_uri=ground_truth_s3, local_path='/opt/ml/processing/input/ground_truth', s3_data_type='S3Prefix')),
]

monitor_outputs = [
    ProcessingOutput(output_name='reports', s3_output=ProcessingS3Output(
        s3_uri=f's3://{bucket_name}/{bucket_prefix}/evidently-reports',
        local_path='/opt/ml/processing/output',
        s3_upload_mode='EndOfJob')),
]

monitor_step = ProcessingStep(
    name='EvidentlyMonitoring',
    step_args=monitor_processor.run(
        code='evidently_monitor.py',
        source_dir='scripts',
        inputs=monitor_inputs,
        outputs=monitor_outputs,
        arguments=[
            '--mlflow-tracking-uri', mlflow_arn,
            '--mlflow-experiment', mlflow_experiment_name,
            '--drift-threshold', p_drift_threshold.to_string(),
            '--lookback-days', p_lookback_days.to_string(),
            '--critical-features', p_critical_features,
            '--min-f1', p_min_f1.to_string(),
            '--sns-topic-arn', p_sns_topic,
            '--endpoint-name', p_endpoint,
        ],
    ),
    cache_config=CacheConfig(enable_caching=False),
)

monitoring_pipeline = Pipeline(
    name=monitoring_pipeline_name,
    parameters=[p_drift_threshold, p_sns_topic, p_endpoint, p_lookback_days, p_critical_features, p_min_f1],
    steps=[monitor_step],
)
print(f'Pipeline defined: {monitoring_pipeline_name}')


### 파이프라인 업서트 및 (선택적으로) 일정 연결

`pipeline.upsert()`는 파이프라인 정의를 생성하거나 업데이트합니다. 선택적 일정은 **Amazon EventBridge Scheduler**를 통해 연결됩니다. `ENABLE_SCHEDULE = True`로 설정하여 활성화합니다. 기본값은 매일 08:00 UTC 실행입니다.

In [ ]:
import json as _json

ENABLE_SCHEDULE = False  # flip to True to activate the daily run
SCHEDULE_EXPRESSION = 'cron(0 8 * * ? *)'  # every day at 08:00 UTC
SCHEDULE_NAME = f'{monitoring_pipeline_name}-daily'

monitoring_pipeline.upsert(role_arn=sm_role)
print(f'Pipeline upserted: {monitoring_pipeline.name}')

if ENABLE_SCHEDULE:
    try:
        scheduler = boto3.client('scheduler', region_name=region)
        pipeline_arn = f'arn:aws:sagemaker:{region}:{boto3.client("sts").get_caller_identity()["Account"]}:pipeline/{monitoring_pipeline.name}'
        scheduler.create_schedule(
            Name=SCHEDULE_NAME,
            ScheduleExpression=SCHEDULE_EXPRESSION,
            FlexibleTimeWindow={'Mode': 'OFF'},
            Target={
                'Arn': pipeline_arn,
                'RoleArn': sm_role,
                'Input': _json.dumps({'PipelineParameterList': []}),
            },
        )
        print(f'Schedule created: {SCHEDULE_NAME} ({SCHEDULE_EXPRESSION})')
    except Exception as e:
        print(f'[WARN] Could not create schedule ({e}). Start the pipeline manually below.')
else:
    print('Schedule disabled (ENABLE_SCHEDULE=False). Start the pipeline manually below.')


### 파이프라인을 수동으로 한 번 실행

지금 실행을 시작하여 엔드투엔드 흐름을 검증합니다. 아래 셀은 파이프라인을 시작하고, SageMaker Studio의 실행 그래프 링크를 렌더링하고, 완료를 기다리고, 스텝을 나열합니다. 그런 다음 MLflow 실행 ID를 출력하여 MLflow UI에서 Evidently 아티팩트로 바로 이동할 수 있습니다.

> **참고:** 데이터 캡처는 엔드포인트 호출 후 S3에 요청을 버퍼링하는 데 1~2분이 걸릴 수 있습니다. 작업이 `캡처 레코드를 찾을 수 없음`으로 실패하면 1분 기다렸다가 셀을 다시 실행하세요.

In [ ]:
from IPython.display import HTML, display

execution = monitoring_pipeline.start(
    parameters={
        'DriftThreshold': 0.05,
        'LookbackDays': 7,              # compare only capture from the last 7 days; set 0 to disable
        'CriticalFeatures': 'duration,age',  # alert individually if any of these drift
        'MinF1': 0.0,                   # set > 0 to also alert on F1 degradation
        'SnsTopicArn': sns_topic_arn or '',
        'EndpointName': endpoint_name,
    }
)
execution_id = execution.describe()['PipelineExecutionArn'].split('/')[-1]
print(f'Pipeline execution started: {execution_id}')

display(HTML(
    f'<b>See <a target="_blank" href="https://studio-{domain_id}.studio.{region}.sagemaker.aws/'
    f'pipelines/{monitoring_pipeline_name}/executions/{execution_id}/graph">the pipeline execution</a> in the Studio UI</b>'
))

%time execution.wait()
execution.list_steps()


In [ ]:
# Show the MLflow run produced by the scheduled monitoring job
from mlflow import MlflowClient

mlflow.set_tracking_uri(mlflow_arn)
client = MlflowClient()
exp = client.get_experiment_by_name(mlflow_experiment_name)
runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="tags.mlflow.runName LIKE 'scheduled_monitoring_%'",
    order_by=['start_time DESC'],
    max_results=1,
)
if runs:
    r = runs[0]
    print(f"MLflow run: {r.info.run_name}")
    print(f"  run_id:    {r.info.run_id}")
    print(f"  share:     {r.data.metrics.get('DriftedColumnsCount.share', 'n/a')}")
else:
    print('No monitoring run found yet — check the Studio pipeline execution logs.')


---

In [ ]:
with mlflow.start_run(run_name=f"comprehensive_monitoring_{timestamp_suffix}") as comp_run:
    print('Creating comprehensive monitoring report...')
    comprehensive_report = Report(metrics=[
        DataDriftPreset(),
        DataSummaryPreset(),
    ])
    comprehensive_report_snapshot = comprehensive_report.run(
        reference_data=reference_data,
        current_data=production_data
    )
    
    comp_filename = f'comprehensive_monitoring_{datetime.now().strftime("%Y%m%d_%H%M%S")}.html'
    comprehensive_report_snapshot.save_html(comp_filename)
    print(f'Comprehensive report saved to: {comp_filename}')

    mlflow.log_artifact(comp_filename, 'evidently_comprehensive_report')
    comp_s3_key = f"{bucket_prefix}/evidently-reports/{comp_filename}"
    s3_client.upload_file(comp_filename, bucket_name, comp_s3_key)
    mlflow.log_param('comprehensive_report_s3', f's3://{bucket_name}/{comp_s3_key}')
    print(f'\nComprehensive monitoring report completed')
    print(f'Report location: s3://{bucket_name}/{comp_s3_key}')

---

## 9: 종합 모니터링 보고서 (선택 사항)

드리프트 감지와 데이터 품질 분석을 결합한 통합 보고서를 생성합니다.

## 10: 요약 및 다음 단계

### 달성한 것

1. **모델 훈련**: MLflow 추적과 함께 SageMaker SDK v3를 사용하여 XGBoost 모델 훈련
2. **실험 추적**: 모든 파라미터, 메트릭, 아티팩트를 SageMaker AI MLflow에 로깅
3. **실시간 추론**: 데이터 캡처가 활성화된 SageMaker 실시간 엔드포인트에 모델 배포
4. **데이터 캡처**: 모니터링을 위해 요청/응답 페이로드를 S3에 자동으로 캡처
5. **분류 평가**: Evidently의 ClassificationPreset을 사용하여 이진 분류 성능 평가
6. **드리프트 감지**: Evidently를 사용하여 훈련 기준과 캡처된 프로덕션 데이터 간의 데이터 드리프트 감지
7. **MLflow 통합**: 모든 분류 메트릭, 드리프트 메트릭, 보고서를 MLflow에 로깅
8. **시각적 보고서**: 분류 성능, 데이터 드리프트, 품질 분석을 위한 인터랙티브 HTML 보고서 생성
9. **예약된 모니터링 파이프라인**: Evidently 검사를 SageMaker 처리 작업으로 래핑하고 SageMaker 파이프라인으로 오케스트레이션
10. **드리프트 알림**: 드리프트된 피처의 비율, 중요 피처, 또는 분류 F1이 설정된 임계값을 초과할 때 SNS 알림 게시

### 다음 단계

1. **이벤트 기반 모니터링**: 시간 기반 일정 대신(또는 함께) 데이터 캡처 접두사의 **S3 이벤트**에서 모니터링 파이프라인을 트리거합니다
2. **폐쇄 루프 재훈련**: 드리프트 또는 F1이 임계값을 위반할 때 재훈련 파이프라인을 시작하는 조건부 스텝으로 파이프라인을 확장합니다
3. **향상된 모니터링**: 피처 드리프트 외에도 예측 분포 드리프트와 비즈니스 KPI를 시간에 따라 추적합니다
4. **대시보드 및 트렌드**: MLflow 메트릭을 기반으로 드리프트 및 품질 트렌드 시각화를 구축합니다
5. **견고성 및 비용**: 룩백 윈도우와 샘플링 비율을 조정하고, 멱등성 재실행을 위한 파이프라인 스텝 캐싱을 활성화합니다

---

In [ ]:
sm_client = boto3.client('sagemaker')

try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f'Deleting endpoint: {endpoint_name} (this may take a minute)')
except Exception as e:
    print(f'Error deleting endpoint: {e}')

try:
    sm_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
    print(f'Deleted endpoint config')
except Exception as e:
    print(f'Error deleting endpoint config: {e}')

try:
    sm_client.delete_model(ModelName=model_name)
    print(f'Deleted model: {model_name}')
except Exception as e:
    print(f'Error deleting model: {e}')

try:
    boto3.client('scheduler', region_name=region).delete_schedule(Name=SCHEDULE_NAME)
    print(f'Deleted schedule: {SCHEDULE_NAME}')
except Exception as e:
    print(f'No schedule to delete or error: {e}')

try:
    monitoring_pipeline.delete()
    print(f'Deleted pipeline: {monitoring_pipeline.name}')
except Exception as e:
    print(f'Error deleting pipeline: {e}')

try:
    if sns_topic_arn:
        sns_client.delete_topic(TopicArn=sns_topic_arn)
        print(f'Deleted SNS topic: {sns_topic_arn}')
except Exception as e:
    print(f'Error deleting SNS topic: {e}')

try:
    reports_prefix = f'{bucket_prefix}/evidently-reports'
    objects = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=reports_prefix).get('Contents', [])
    if objects:
        s3_client.delete_objects(Bucket=bucket_name, Delete={'Objects': [{'Key': o['Key']} for o in objects]})
        print(f'Deleted {len(objects)} Evidently report objects from s3://{bucket_name}/{reports_prefix}')
except Exception as e:
    print(f'Error deleting Evidently reports: {e}')

print('\nNote: MLflow runs and S3 data are preserved for historical analysis.')

---